# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2 dataset) Exploration with `mlcroissant`

This notebook demonstrates step-by-step data exploration using the [mlcroissant](https://github.com/mlcommons/croissant) library for machine-actionable datasets.

### Dataset Source
FAIR^2: Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors, including MSI-H Status and Anatomical Distribution

- **Croissant schema URL**: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- **Identifier**: 10.71728/senscience.qs2f-h81p

The source dataset contains multiple record sets describing clinical and pathological variables from 77 cancer survivors, such as demographics, comorbidities, primary & secondary cancer types, and biomarker status.

In [ ]:
# Ensure mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading

We'll load both the dataset *metadata* and the records using `mlcroissant`. This enables structured access to the record sets and fields via their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset loaded: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Dataset identifier: {metadata.identifier}")

## 2. Data Overview

Let's review the record sets, fields, columns, and examine their `@id` values for structured access.

> **Note:** In Croissant, the main table-like entities are `RecordSet`s, which contain `Field`s and `Column`s. Each is uniquely referenced by its `@id`.

In [ ]:
# List all record sets in the dataset with their @id and name.
if not hasattr(dataset, 'record_sets'):
    record_sets = dataset.metadata._json_dict.get('recordSet', [])
else:
    record_sets = dataset.record_sets

if not record_sets:
    # Alternative fallback: mlcroissant >=0.4 returns Dataset.record_sets
    # Rarely, recordSet may not be set; fallback to scanning metadata._json_dict
    print('No record sets found in the dataset metadata.')
else:
    print('Record sets available:')
    overview = []
    for rs in record_sets:
        if isinstance(rs, str):
            rid = rs
            rmeta = None
        elif hasattr(rs, '@id'):
            rid = rs['@id'] if isinstance(rs, dict) else rs['@id']
            rmeta = rs
        else:
            # For mlcroissant.record_sets (object)
            rid = rs.id
            rmeta = rs
        print(f"- RecordSet ID: {rid}")
        overview.append(rid)
    if not overview and hasattr(dataset, 'record_sets'):
        for rs in dataset.record_sets:
            print(f"- RecordSet ID: {rs.id}, Name: {rs.name if hasattr(rs, 'name') else ''}")
            overview.append(rs.id)

# Short version: Display all fields and columns within the first available record set

if hasattr(dataset, 'record_sets') and len(dataset.record_sets) > 0:
    first_rs = dataset.record_sets[0]
    print(f"\nFirst RecordSet ID: {first_rs.id}, Name: {first_rs.name if hasattr(first_rs, 'name') else ''}")
    print('Fields:')
    for field in first_rs.fields:
        fname = getattr(field, 'name', getattr(field, 'id', ''))
        print(f"\tField @id: {field.id} \tname: {fname}")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"\t\tColumn @id: {col.id} (name: {col.name})")

## 3. Data Extraction

We'll load data from the dataset's main record set(s) using their `@id` values, and convert the records into pandas DataFrames for inspection and analysis.

**Note:** The example dataset typically has one main record set representing the primary table. We'll use its `@id` below.

In [ ]:
# Get the available record set IDs programmatically
record_set_ids = [rs.id for rs in dataset.record_sets]
print("RecordSet @ids:", record_set_ids)

# We'll use the first record set for demonstration; update index if others are desired
main_record_set_id = record_set_ids[0]
print(f"\nLoading records from record set: {main_record_set_id}")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only create DataFrame if data is present
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for RecordSet @id: {record_set_id}")
    else:
        print(f"No records found in RecordSet @id: {record_set_id}")

print("\nColumns in main DataFrame:")
if main_record_set_id in dataframes:
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print(f"No DataFrame available for RecordSet @id: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Let's perform simple data processing:
- Filter records using a numeric field (e.g., *Age*, if available)
- Normalize a numeric field
- Group by a key attribute (e.g., *Sex*, *MSI_Status*)

**All fields/columns are referenced by their unique `@id`.**

> Replace the variable names below with the actual `@id` strings from your dataset columns for correct operation.

In [ ]:
# Example: Assume there is a numeric field with @id 'age' and a groupable field 'sex' (update to match actual @id)
# For demonstration, we'll attempt to detect likely field names programmatically

df = dataframes[main_record_set_id]

# Find likely numeric columns (e.g., Age, Intervals, Counts)
numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi' or 'age' in col.lower() or 'interval' in col.lower()]
print(f"Numeric field candidates: {numeric_candidates}")
if numeric_candidates:
    numeric_field = numeric_candidates[0]  # Default to first candidate
else:
    raise ValueError("No numeric field found!")

# Find a likely field for grouping, e.g., 'sex', 'MSI_Status', or similar categorical variable
group_candidates = [col for col in df.columns if any(g in col.lower() for g in ['sex', 'gender', 'msi', 'group'])]
print(f"Group-by field candidates: {group_candidates}")
group_field = group_candidates[0] if group_candidates else None

# Filter for demonstration: All records with numeric_field > threshold
threshold = 50 if 'age' in numeric_field.lower() else df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold].copy()

print(f"\nFiltered records with {numeric_field} > {threshold}: ({filtered_df.shape[0]} rows)")
print(filtered_df.head(3))

# Normalize the numeric field
norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, norm_col]].head(3))

# If group_field available, show group statistics
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field} by {group_field}:")
    print(grouped_df)

## 5. Visualization

Here, we visualize data distributions and group-based comparisons. For instance, we visualize the (normalized) distribution of a numeric field and its relation to a grouping (if available).

You can easily extend this to any column by referencing its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field (histogram)
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field], kde=True, bins=10, color='skyblue')
plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If group_field available, plot group-wise boxplot
if group_field:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field], palette='Set2')
    plt.title(f"{numeric_field} by {group_field}")
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, examine, and process a clinical dataset described with a Croissant schema. Using only `@id` references for record sets and fields, we've:
- Inspected metadata, available record sets, and their fields/columns.
- Loaded main table records as a pandas DataFrame.
- Performed simple filtering, normalization, and group-wise summaries referencing fields by their `@id`.
- Visualized numeric field distributions and categorical/binary groupings, supporting further statistical or ML analysis.

For further domain-specific exploration, see the [FAIR^2 dataset documentation](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and adapt field/group selection as required for your questions.